In [3]:
#!pip install -U -q "google-genai==1.7.0"

In [1]:
from google import genai
from google.genai import types

from IPython.display import HTML, Markdown, display

In [3]:
with open("../../gemini_key.txt","r") as f:
    string = f.read()

GOOGLE_API_KEY = string

client = genai.Client(api_key=GOOGLE_API_KEY)

In [5]:
## Prompting Gemini 2.0 Flash
response = client.models.generate_content(
    model="gemini-2.0-flash",
    contents="Explain what is Gemini 2.0 model in 200 words")

print(response.text)

Gemini 2.0 is Google's next generation of their Gemini AI model, designed to be even more powerful and versatile than its predecessor. While specific details are still emerging, key improvements are expected in several areas.

**Enhanced Multimodality:** Gemini 2.0 will likely further blur the lines between different data types. Expect improved capabilities in understanding and generating text, images, audio, video, and code, allowing it to seamlessly integrate and manipulate information across modalities.

**Improved Reasoning and Problem Solving:** A significant focus is on advancing reasoning abilities. This means Gemini 2.0 should be better at tackling complex problems, making logical inferences, and understanding nuanced contexts.

**Expanded Context Window:** One anticipated advancement is a much larger context window, enabling the model to process and retain more information over longer conversations and analyses. This will lead to more coherent and contextually relevant outputs

In [7]:
# Create a Chat with Gemini
chat = client.chats.create(model='gemini-2.0-flash', history=[])
response = chat.send_message("""Hi , I am Anshul, I am creating this chat to understand more about the world. 
Can you tell me about the first GPT1 model. Limit your response to 100 words""")
print(response.text)

Hi Anshul! I'm happy to help you learn about the world.

GPT-1 (Generative Pre-trained Transformer) was introduced by OpenAI in 2018. Its key innovation was using a transformer architecture pre-trained on a large corpus of text (BooksCorpus) and then fine-tuned for specific tasks. This allowed it to achieve strong performance on various natural language processing tasks like question answering and text classification with minimal task-specific training data. It was a pivotal step in demonstrating the power of large language models.



In [9]:
response = chat.send_message("Great!, how did it compare against GPT2? give me the top 3 bullet points and make it concise") 

print(response.text)

Here are the top 3 differences between GPT-1 and GPT-2:

*   **Scale:** GPT-2 was much larger, with 1.5 billion parameters compared to GPT-1's 117 million, enabling better performance.
*   **Zero-Shot Learning:** GPT-2 demonstrated zero-shot learning, performing tasks without specific fine-tuning. GPT-1 required task-specific fine-tuning.
*   **Generation Quality:** GPT-2 produced significantly more coherent and contextually relevant text than GPT-1, although still with limitations like repetition.



## Let's explore some generation parameters that can change the LLM response

### We can try out following parameters - 
1. Temparature - Determines the randomness in LLM's output (higher the temperature, more randomness). This can ensure slight variation in output when same prompt is asked multiple times
2. Top-P - It considers all top tokens till their cumulative probability reaches P. This can also enable variation in outputs
3. max_output_tokens - Limits the response to that many tokens (truncates instead of making the response concise)

In [11]:
from google.genai import types

In [13]:
## Temperature

high_temp_config = types.GenerateContentConfig(temperature=2.0)


for _ in range(5):
  response = client.models.generate_content(
      model='gemini-2.0-flash',
      config=high_temp_config,
      contents='Pick a random animal (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Quokka
 -------------------------
Echidna
 -------------------------
Axolotl
 -------------------------
Axolotl
 -------------------------
Squirrel
 -------------------------


In [15]:
low_temp_config = types.GenerateContentConfig(temperature=0.0)


for _ in range(5):
  response = client.models.generate_content(
      model='gemini-2.0-flash',
      config=low_temp_config,
      contents='Pick a random animal (respond in a single word)')

  if response.text:
    print(response.text, '-' * 25)

Axolotl
 -------------------------
Axolotl
 -------------------------
Axolotl
 -------------------------
Elephant
 -------------------------
Elephant
 -------------------------


In [19]:
## Top-P

model_config = types.GenerateContentConfig(
    # These are the default values for gemini-2.0-flash.
    temperature=1.0,
    top_p=0.95,
)



for i in range(2):
    story_prompt = "You are a creative writer. Write a haiku describing the beaty of Japan"
    response = client.models.generate_content(
        model='gemini-2.0-flash',
        config=model_config,
        contents=story_prompt)
    print("Attempt 1: " + str(i+1))
    print(response.text)

Attempt 1: 1
Green hills touch the sky,
Cherry blossoms paint the air,
Ancient peace remains.

Attempt 1: 2
Green hills meet blue sky,
Cherry blossoms softly fall,
Ancient peace remains. 



In [21]:
## Max output tokens
## You can that the response gets truncated to 100 tokens, this can be used in some cases to reduce cost

model_config = types.GenerateContentConfig(
    # These are the default values for gemini-2.0-flash.
    max_output_tokens = 100
)


story_prompt = "You are a creative writer. Write an essay on Japan in 500 words"
response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=model_config,
    contents=story_prompt)
print(response.text)

Japan, a land where ancient traditions dance with futuristic innovation, is a captivating tapestry woven from silk threads of serenity and electrifying currents of progress. It's a place where bowing geishas glide past neon-drenched arcades, where the hushed reverence of a Shinto shrine echoes against the rhythmic rumble of a bullet train, and where the delicate art of calligraphy finds its counterpart in the precision of robotics.

The very soul of Japan seems to breathe in the stillness of its meticulously crafted


## Types of Prompting

#### Zero-Shot prompting

In [34]:
model_config = types.GenerateContentConfig(
    temperature=0.1,
    top_p=1,
    max_output_tokens=5,
)

zero_shot_prompt = """Classify product reviews as POSITIVE, NEUTRAL or NEGATIVE.
Review: "This vaccuum cleaner works slightly worse than advertised. It is very loud and the suction is not as strong."
Sentiment: """

response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=model_config,
    contents=zero_shot_prompt)

print(response.text)

NEGATIVE



In [36]:
## Enum mode

import enum

class Sentiment(enum.Enum):
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    NEGATIVE = "negative"


response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=types.GenerateContentConfig(
        response_mime_type="text/x.enum",
        response_schema=Sentiment
    ),
    contents=zero_shot_prompt)

print(response.text)

negative


#### Few-shot prompting

In [39]:
few_shot_prompt = """Parse a customer's grocery order into valid JSON:

EXAMPLE:
Please order paneer, tomato and coke from Instamart tomorrow
JSON Response:
```
{
"source": "Instamart",
"time": "tomorrow",
"items": ["paneer", "tomato", "coke"]
}
```

EXAMPLE:
Can you order a zucchini, broccoli, a yogurt cup and two icream cones from Zepto today at 6 pm
JSON Response:
```
{
"source": "Zepto",
"time": "today at 6 pm",
"items": ["a zucchini", "broccoli", "a yogurt cup","two icream cones"]
}
```

ORDER:
"""

customer_order = "Get me 1 kg atta, 2 packets of milk and ghee from Blinkit this weekend"

response = client.models.generate_content(
    model='gemini-2.0-flash',
    config=types.GenerateContentConfig(
        temperature=0.1,
        top_p=1,
        max_output_tokens=250,
    ),
    contents=[few_shot_prompt, customer_order])

print(response.text)

```json
{
"source": "Blinkit",
"time": "this weekend",
"items": ["1 kg atta", "2 packets of milk", "ghee"]
}
```


#### Chain of Thought (CoT) prompting

In [48]:
## Without chain of though
prompt = """When I was 4 years old, my partner was 3 times my age. Now, I
am 20 years old. How old is my partner? Return the answer directly."""

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt)

print(response.text)

68



In [50]:
## With chain of though
prompt = """When I was 4 years old, my partner was 3 times my age. Now, I
am 20 years old. How old is my partner? Think of it step by step."""

response = client.models.generate_content(
    model='gemini-2.0-flash',
    contents=prompt)

print(response.text)

Here's how to solve this step-by-step:

1. **Find the age difference:** When you were 4, your partner was 3 times your age, meaning they were 4 * 3 = 12 years old.

2. **Calculate the age difference:** The age difference between you and your partner is 12 - 4 = 8 years.

3. **Determine your partner's current age:** Since the age difference remains the same, your partner is currently 20 + 8 = 28 years old.

**Therefore, your partner is currently 28 years old.**

